# 02 — The served model (TF-IDF backbone)

**Spec: §7, adapted.** The spec's Phase 3 fine-tunes CamemBERT. That needs a
GPU. This notebook trains the **TF-IDF + LogisticRegression** model instead and
promotes it to the *served* model, so every downstream phase — calibration, the
neutral band, the aggregate correction, aspects, the demo — can be built and
validated end to end today.

**This is a backbone swap, not a shortcut.** `src/predict.py` defines the
pipeline once and takes the encoder as a backend. When CamemBERT is trained,
you change two fields in `inference_config.json` and re-run notebooks 04 and
06; nothing else moves. The architecture is what makes that safe — if the
transformer had its own copy of the pipeline, the two would drift and the
published numbers would stop describing the deployed system.

Carry this honestly into the README: the shipped system currently has a
**0.9286 macro-F1 bag-of-ngrams** backbone, not a transformer.

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from src.normalize import normalize, NORMALIZE_VERSION
import joblib
from datasets import load_dataset
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

ds = load_dataset("tblard/allocine")
ds.pop("test")                      # §1.1 -- test does not exist here

sub_idx = np.load(REPO / "data" / "train_subsample_40k.npy")
Xtr = [normalize(t) for t in ds["train"]["review"]]
ytr = np.array(ds["train"]["label"])
Xtr, ytr = [Xtr[i] for i in sub_idx], ytr[sub_idx]
Xva = [normalize(t) for t in ds["validation"]["review"]]
yva = np.array(ds["validation"]["label"])
print(f"train {len(Xtr):,} | validation {len(Xva):,}")

In [ ]:
t0 = time.perf_counter()
model = make_pipeline(
    FeatureUnion([
        ("w", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("c", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                              min_df=3, sublinear_tf=True)),
    ]),
    LogisticRegression(max_iter=3000, class_weight="balanced"),
).fit(Xtr, ytr)
print(f"trained in {time.perf_counter() - t0:.1f}s")

pred = model.predict(Xva)
print(classification_report(yva, pred, target_names=["négatif", "positif"],
                            digits=4))

## Split validation in two

Everything downstream needs labelled data the model did not train on, and
those uses must not contaminate each other:

* **calib half** — fits the temperature and the neutral band (notebook 04)
* **holdout half** — measures TPR/FPR for the correction (notebook 06)

Using one half for both would make the measured TPR/FPR reflect thresholds
tuned on the same rows, and the aggregate interval would come out too narrow.

In [ ]:
rng = np.random.default_rng(42)
perm = rng.permutation(len(Xva))
calib_idx, hold_idx = perm[: len(perm) // 2], perm[len(perm) // 2 :]

np.save(REPO / "data" / "calib_idx.npy", calib_idx)
np.save(REPO / "data" / "hold_idx.npy", hold_idx)

model_path = REPO / "data" / "model_tfidf.joblib"
joblib.dump(model, model_path, compress=3)
size_mb = model_path.stat().st_size / 1e6
print(f"calib {len(calib_idx):,} | holdout {len(hold_idx):,}")
print(f"saved {model_path.name}  ({size_mb:.1f} MB)")

In [ ]:
cfg_path = REPO / "inference_config.json"
cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
cfg["backend"] = "sklearn"
cfg["model_repo"] = str(model_path)
cfg["revision"] = "local-tfidf-v1"
cfg["labels"] = ["négatif", "positif"]
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False) + "\n",
                    encoding="utf-8")

json.dump({"accuracy": round(float(accuracy_score(yva, pred)), 4),
           "macro_f1": round(float(f1_score(yva, pred, average="macro")), 4),
           "size_mb": round(size_mb, 2)},
          open(REPO / "data" / "results_model.json", "w"), indent=2)
print(cfg_path.read_text(encoding="utf-8"))